In [1]:
import pandas as pd
import numpy as np
import joblib
from tqdm import tqdm
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler

from torch.utils.data import DataLoader,TensorDataset

In [2]:
df = pd.read_csv("train.csv")

In [3]:
test_df = pd.read_csv("test.csv")

In [4]:
df.head(2)

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0


In [5]:
ohe = OneHotEncoder(drop="first",handle_unknown="ignore",sparse_output=False)
encoder = ohe.fit_transform(df[["Sex"]])

In [6]:
ohe.get_feature_names_out()

array(['Sex_male'], dtype=object)

In [7]:
joblib.dump(ohe,filename="OneHotEncoder.pkl")

['OneHotEncoder.pkl']

In [8]:
encoder_df_1 = pd.DataFrame(encoder,columns=ohe.get_feature_names_out())

In [9]:
final_df = pd.concat(
    [df.drop("Sex",axis=1),encoder_df_1],axis=1
)

In [10]:
final_df

,id,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,Sex_male
0,0,36,189.0,82.0,26.0,101.0,41.0,150.0,1.0
1,1,64,163.0,60.0,8.0,85.0,39.7,34.0,0.0
2,2,51,161.0,64.0,7.0,84.0,39.8,29.0,0.0
3,3,20,192.0,90.0,25.0,105.0,40.7,140.0,1.0
4,4,38,166.0,61.0,25.0,102.0,40.6,146.0,0.0
...,...,...,...,...,...,...,...,...,...
749995,749995,28,193.0,97.0,30.0,114.0,40.9,230.0,1.0
749996,749996,64,165.0,63.0,18.0,92.0,40.5,96.0,0.0
749997,749997,60,162.0,67.0,29.0,113.0,40.9,221.0,1.0
749998,749998,45,182.0,91.0,17.0,102.0,40.3,109.0,1.0


In [11]:
x = final_df.drop(["id","Calories"],axis=1).values
y = final_df["Calories"].values

In [12]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [13]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [14]:
scaler_y_train = StandardScaler()
y_train = scaler_y_train.fit_transform(y_train.reshape(-1,1))
y_test = scaler_y_train.transform(y_test.reshape(-1,1))

In [15]:
joblib.dump(scaler,filename="x_train_x_test_scaler.pkl")
joblib.dump(scaler_y_train,filename="y_train_y_test_scaler.pkl")

['y_train_y_test_scaler.pkl']

In [16]:
x_train = torch.tensor(x_train,dtype=torch.float32)
x_test = torch.tensor(x_test,dtype=torch.float32)

In [17]:
y_train = torch.tensor(y_train,dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test,dtype=torch.float32).view(-1,1)

In [18]:
train_dataset = TensorDataset(
    x_train,y_train
)

test_dataset = TensorDataset(
    x_test,
    y_test
)

In [3]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=True
)

NameError: name 'DataLoader' is not defined

In [28]:
train_loader.num_workers

0

In [ ]:
model = nn.Sequential(
    nn.Linear(in_features=7,out_features=128),
    #nn.BatchNorm1d(128),
    nn.LayerNorm(128),
    nn.Mish(),
    nn.Dropout(0.2),

    nn.Linear(in_features=128,out_features=64),
    #nn.BatchNorm1d(64),
    nn.LayerNorm(64),
    nn.Mish(),
    nn.Dropout(0.2),
    
    nn.Linear(64,32),
    #nn.BatchNorm1d(32),
    nn.LayerNorm(32),
    nn.Mish(),
    nn.Dropout(0.3),

    nn.Linear(32,1)
)

In [50]:
loss_function = nn.MSELoss()

optimizer_function = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs = 250
process_bar = tqdm(range(epochs), colour="green", desc="Ann_Model")

for i in process_bar:
    # --- 1. TRAINING PHASE ---
    model.train()
    total_train_loss = 0
    
    for batch_x, batch_y in train_loader:
        optimizer_function.zero_grad()
        prediction = model(batch_x)
        loss = loss_function(prediction, batch_y)
        loss.backward()
        optimizer_function.step()
        
        # Sahi loss accumulation
        total_train_loss += loss.item()
    
    # Average Training Loss nikalne ke liye total batches se divide karein
    avg_train_loss = total_train_loss / len(train_loader)

    # --- 2. VALIDATION PHASE ---
    model.eval() # Model ko evaluation mode mein dalein
    total_val_loss = 0
    
    with torch.no_grad(): # Gradient calculate nahi karna hai
        for val_x, val_y in val_loader: # Aapka validation data loader
            val_prediction = model(val_x)
            val_loss = loss_function(val_prediction, val_y)
            total_val_loss += val_loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader)

    # --- 3. CHECKPOINT SAVE (Loop ke baahar, har epoch ke baad) ---
    torch.save({
        "Epoch": i,
        "checkpoints": model.state_dict(),
        "Train_Loss": avg_train_loss,
        "Val_Loss": avg_val_loss,
        "Optimizer": optimizer_function.state_dict(),
    }, f"models/checkpoint_{i}.pth")

    # --- 4. PROGRESS BAR UPDATE ---
    process_bar.set_postfix({
        "Epoch": i,
        "Train_Loss": f"{avg_train_loss:.4f}",
        "Val_Loss": f"{avg_val_loss:.4f}"
    })


Ann_Model::  46%|████▌     | 114/250 [2:48:50<3:32:14, 93.64s/it, Epochs=114, Loss=0.0184]

In [2]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X, y in test_loader:

        outputs = model(X)

        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y).sum().item()

        total += y.size(0)

accuracy = correct / total

print("Accuracy:", accuracy)

NameError: name 'model' is not defined